In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

carpark2609 = pd.read_csv("CA1_Datasets/carpark_20260901_20260930.csv",
    dtype={
        "carpark_number": "string",
        "lot_type": "string"
    }
)

bus_stops2609 = pd.read_csv("CA1_Datasets/bus_stops.csv",
    dtype={
        "BusStopCode": "string",
        "RoadName": "string",
        "Description": "string"
    }
)

od_bus2609 = pd.read_csv("CA1_Datasets/origin_destination_bus_202609.csv",
    dtype={
        "YEAR_MONTH": "string",
        "DAY_TYPE": "string",
        "PT_TYPE": "string",
        "ORIGIN_PT_CODE": "string",
        "DESTINATION_PT_CODE": "string"
    }
)

rainfall_2609 = pd.read_csv("CA1_Datasets/rainfall_20260901_20260930.csv",
    dtype={
        "station_id": "string",
        "station_name": "string"
    }
)

datasets = {"carpark": carpark2609, "bus_stops": bus_stops2609, "bus_od": od_bus2609, "rainfall": rainfall_2609}

In [2]:
def validate_and_quarantine(df, validation_rules):
    df = df.copy()
    # Store all applicable validation reasons.
    reasons = []

    for reason_code, invalid_mask in validation_rules.items():
        reasons.append(
            pd.Series(
                np.where(invalid_mask, reason_code, ""),
                index=df.index
            )
        )
    reason_df = pd.concat(reasons, axis=1)

    df["quarantine_reason"] = reason_df.apply(
        lambda row: "|".join(
            value for value in row if value != ""
        ),
        axis=1
    )
    quarantine = df[df["quarantine_reason"] != ""].copy()
    clean = df[df["quarantine_reason"] == ""].copy()

    clean.drop(columns=["quarantine_reason"], inplace=True)

    return clean, quarantine


In [3]:
OUTPUT_DIR = Path("processed_data")
QUARANTINE_DIR = OUTPUT_DIR / "quarantine"
REPORT_DIR = OUTPUT_DIR / "reports"

OUTPUT_DIR.mkdir(exist_ok=True)
QUARANTINE_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

carpark_raw = carpark2609.copy()
bus_stops_raw = bus_stops2609.copy()
bus_od_raw = od_bus2609.copy()
rainfall_raw = rainfall_2609.copy()

In [4]:
for name, df in datasets.items():

    print(f"\n{'=' * 60}")
    print(f"Dataset {name.upper()}")
    print(f"{'=' * 60}")

    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nBlank string values:")
    print(
        df.select_dtypes(include=["object", "string"])
        .apply(lambda col: col.str.strip().eq("").sum())
    )

    print("\nDuplicate rows:")
    print(df.duplicated().sum())


Dataset CARPARK
Rows: 1,745,692
Columns: 6

Data types:
snapshot_time              object
carpark_number     string[python]
lot_type           string[python]
lots_available              int64
total_lots                  int64
update_datetime            object
dtype: object

Missing values:
snapshot_time      0
carpark_number     0
lot_type           0
lots_available     0
total_lots         0
update_datetime    0
dtype: int64

Blank string values:
snapshot_time      0
carpark_number     0
lot_type           0
update_datetime    0
dtype: int64

Duplicate rows:
0

Dataset BUS_STOPS
Rows: 5,209
Columns: 5

Data types:
BusStopCode    string[python]
RoadName       string[python]
Description    string[python]
Latitude              float64
Longitude             float64
dtype: object

Missing values:
BusStopCode    0
RoadName       0
Description    0
Latitude       0
Longitude      0
dtype: int64

Blank string values:
BusStopCode    0
RoadName       0
Description    0
dtype: int64

Duplicate 

### Check duplicate analytical keys

In [5]:
# Bus OD
bus_od_key = [
    "YEAR_MONTH",
    "DAY_TYPE",
    "TIME_PER_HOUR",
    "PT_TYPE",
    "ORIGIN_PT_CODE",
    "DESTINATION_PT_CODE"
]

# Rainfall
rainfall_key = [
    "station_id",
    "timestamp"
]

# Carpark
carpark_key = [
    "carpark_number",
    "lot_type",
    "update_datetime"
]

# Bus stops
bus_stop_key = [
    "BusStopCode"
]

key_checks = {
    "Bus OD": (od_bus2609, bus_od_key),
    "Rainfall": (rainfall_2609, rainfall_key),
    "Carpark": (carpark2609, carpark_key),
    "Bus Stops": (bus_stops2609, bus_stop_key)
}

for name, (df, keys) in key_checks.items():

    duplicate_mask = df.duplicated(
        subset=keys,
        keep=False
    )

    duplicate_records = df[duplicate_mask]

    print(f"\n{name}")
    print(f"Duplicate key rows: {len(duplicate_records):,}")
    print(
        f"Duplicate key groups: "
        f"{df.duplicated(subset=keys).sum():,}"
    )


Bus OD
Duplicate key rows: 0
Duplicate key groups: 0

Rainfall
Duplicate key rows: 0
Duplicate key groups: 0

Carpark
Duplicate key rows: 21,223
Duplicate key groups: 16,555

Bus Stops
Duplicate key rows: 0
Duplicate key groups: 0


### Check invalid timestamp formats

In [6]:
print("BUS OD")
print("YEAR_MONTH:", od_bus2609["YEAR_MONTH"].unique()[:15])
print("DAY_TYPE:", od_bus2609["DAY_TYPE"].unique())
print("TIME_PER_HOUR:", od_bus2609["TIME_PER_HOUR"].unique())
print("PT_TYPE:", od_bus2609["PT_TYPE"].unique())

print("\nCARPARK")
print("LOT_TYPE:", carpark2609["lot_type"].unique())

print("\nRAINFALL")
print("Rainfall range:")
print(rainfall_2609["rainfall_mm"].describe())

BUS OD
YEAR_MONTH: <StringArray>
['2026-09']
Length: 1, dtype: string
DAY_TYPE: <StringArray>
['WEEKDAY', 'WEEKENDS/HOLIDAY']
Length: 2, dtype: string
TIME_PER_HOUR: [19 12 10 18 11  0  8  6 17  7 21 16 23 13 20 22 14 15  5  9  1  2  4  3]
PT_TYPE: <StringArray>
['BUS']
Length: 1, dtype: string

CARPARK
LOT_TYPE: <StringArray>
['C', 'Y', 'H', 'S']
Length: 4, dtype: string

RAINFALL
Rainfall range:
count    761445.000000
mean          0.023575
std           0.317894
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          19.700000
Name: rainfall_mm, dtype: float64


In [7]:
def numeric_profile(df, column, name):

    values = pd.to_numeric(df[column], errors="coerce")

    invalid = values.isna()
    print(f"\n{name} - {column}")
    print(f"Unparseable values: {invalid.sum():,}")

    return values


# Bus OD
trips = numeric_profile(
    od_bus2609, "TOTAL_TRIPS", "Bus OD"
)

print("Negative passenger counts:", (trips < 0).sum())


# Rainfall
rain = numeric_profile(
    rainfall_2609, "rainfall_mm", "Rainfall"
)

print("Negative rainfall values:", (rain < 0).sum())


# Carpark
available = numeric_profile(
    carpark2609, "lots_available", "Carpark"
)

capacity = numeric_profile(
    carpark2609, "total_lots", "Carpark"
)

invalid_capacity = (
    (capacity <= 0) |
    (available < 0) |
    (available > capacity)
)

print(
    "Invalid parking capacity/availability records:",
    invalid_capacity.sum()
)


Bus OD - TOTAL_TRIPS
Unparseable values: 0
Negative passenger counts: 0

Rainfall - rainfall_mm
Unparseable values: 0
Negative rainfall values: 0

Carpark - lots_available
Unparseable values: 0

Carpark - total_lots
Unparseable values: 0
Invalid parking capacity/availability records: 3908


In [8]:
timestamp_columns = {
    "Carpark snapshot": (carpark2609, "snapshot_time"),
    "Carpark source update": (carpark2609, "update_datetime"),
    "Rainfall observation": (rainfall_2609, "timestamp")
}

for name, (df, column) in timestamp_columns.items():

    parsed = pd.to_datetime(
        df[column],
        errors="coerce",
        utc=True
    )

    print(f"\n{name}")
    print("Unparseable timestamps:", parsed.isna().sum())
    print("Earliest:", parsed.min())
    print("Latest:", parsed.max())


Carpark snapshot
Unparseable timestamps: 0
Earliest: 2026-08-31 16:00:37+00:00
Latest: 2026-09-30 15:00:36+00:00

Carpark source update
Unparseable timestamps: 0
Earliest: 2025-11-01 13:45:00+00:00
Latest: 2026-09-30 23:01:04+00:00

Rainfall observation
Unparseable timestamps: 0
Earliest: 2026-08-31 16:00:00+00:00
Latest: 2026-09-30 15:55:00+00:00


## Exploring carpark dataset issues
- Duplicates
- Invalid records
- Unique timestamp records

In [9]:
# Inspect duplicate carpark key groups
carpark_key = ["carpark_number", "lot_type", "update_datetime"]

duplicate_mask = carpark2609.duplicated(
    subset=carpark_key,
    keep=False
)

carpark_duplicates = carpark2609[duplicate_mask].copy()

# Count how many distinct values exist within each duplicate key
duplicate_summary = (
    carpark_duplicates
    .groupby(carpark_key)
    .agg(
        row_count=("snapshot_time", "size"),
        distinct_snapshots=("snapshot_time", "nunique"),
        distinct_available=("lots_available", "nunique"),
        distinct_capacity=("total_lots", "nunique")
    )
    .reset_index()
)

print("Duplicate groups:", len(duplicate_summary))
print("\nValue variation within duplicate groups:")
print(
    duplicate_summary[
        ["distinct_snapshots", "distinct_available", "distinct_capacity"]
    ].value_counts().head(20)
)

print("\nGroups with conflicting availability or capacity:")
print(
    duplicate_summary[
        (duplicate_summary["distinct_available"] > 1) |
        (duplicate_summary["distinct_capacity"] > 1)
    ].head(20)
)

Duplicate groups: 4668

Value variation within duplicate groups:
distinct_snapshots  distinct_available  distinct_capacity
2                   1                   1                    3202
3                   1                   1                     713
4                   1                   1                     357
5                   1                   1                     131
6                   1                   1                      63
7                   1                   1                      43
8                   1                   1                      29
10                  1                   1                      21
9                   1                   1                      20
12                  1                   1                      18
11                  1                   1                      12
720                 1                   1                       8
20                  1                   1                       7
16                 

In [10]:
# Break down invalid carpark records by reason
available = pd.to_numeric(carpark2609["lots_available"], errors="coerce")
capacity = pd.to_numeric(carpark2609["total_lots"], errors="coerce")

invalid_reasons = pd.DataFrame({
    "capacity_zero_or_negative": capacity <= 0,
    "availability_negative": available < 0,
    "availability_exceeds_capacity": available > capacity,
    "capacity_missing_or_unparseable": capacity.isna(),
    "availability_missing_or_unparseable": available.isna()
})

print("Invalid records by condition:")
print(invalid_reasons.sum())

invalid_mask = invalid_reasons.any(axis=1)

print("\nTotal invalid records:", invalid_mask.sum())

print("\nSample invalid records:")
display(
    carpark2609.loc[
        invalid_mask,
        [
            "snapshot_time", "carpark_number", "lot_type",
            "lots_available", "total_lots", "update_datetime"
        ]
    ].head(20)
)

Invalid records by condition:
capacity_zero_or_negative              3188
availability_negative                     0
availability_exceeds_capacity           720
capacity_missing_or_unparseable           0
availability_missing_or_unparseable       0
dtype: int64

Total invalid records: 3908

Sample invalid records:


,snapshot_time,carpark_number,lot_type,lots_available,total_lots,update_datetime
757,2026-09-01T00:00:37+08:00,PL40,C,0,0,2026-08-04T00:00:00
966,2026-09-01T00:00:37+08:00,JM33,S,0,0,2026-05-07T14:39:09
1851,2026-09-01T00:00:37+08:00,PL57,S,0,0,2026-04-16T20:27:39
1971,2026-09-01T00:00:37+08:00,A88,Y,64,46,2026-08-31T23:51:14
2201,2026-09-01T00:00:37+08:00,A33,C,0,0,2026-08-31T23:59:35
3181,2026-09-01T01:00:37+08:00,PL40,C,0,0,2026-08-04T00:00:00
3388,2026-09-01T01:00:37+08:00,JM33,S,0,0,2026-05-07T14:39:09
4268,2026-09-01T01:00:37+08:00,PL57,S,0,0,2026-04-16T20:27:39
4392,2026-09-01T01:00:37+08:00,A88,Y,64,46,2026-09-01T00:45:47
4624,2026-09-01T01:00:37+08:00,A33,C,0,0,2026-09-01T01:00:06


In [11]:
print("Carpark snapshot_time:")
print(carpark2609["snapshot_time"].head(5).to_list())

print("\nCarpark update_datetime:")
print(carpark2609["update_datetime"].head(10).to_list())

print("\nRainfall timestamp:")
print(rainfall_2609["timestamp"].head(5).to_list())

Carpark snapshot_time:
['2026-09-01T00:00:37+08:00', '2026-09-01T00:00:37+08:00', '2026-09-01T00:00:37+08:00', '2026-09-01T00:00:37+08:00', '2026-09-01T00:00:37+08:00']

Carpark update_datetime:
['2026-08-31T23:59:45', '2026-09-01T00:00:04', '2026-08-31T23:59:46', '2026-09-01T00:00:07', '2026-08-31T23:59:59', '2026-09-01T00:00:10', '2026-09-01T00:00:20', '2026-08-31T23:59:56', '2026-09-01T00:00:08', '2026-08-31T23:59:40']

Rainfall timestamp:
['2026-09-01T23:55:00+08:00', '2026-09-01T23:55:00+08:00', '2026-09-01T23:55:00+08:00', '2026-09-01T23:55:00+08:00', '2026-09-01T23:55:00+08:00']


### number of unique timestamps and stations

In [12]:
# Carpark snapshot coverage
carpark_snapshots = pd.to_datetime(
    carpark2609["snapshot_time"],
    errors="coerce"
)

print("CARPARK SNAPSHOT COVERAGE")
print("Unique snapshots:", carpark_snapshots.nunique())
print("Earliest:", carpark_snapshots.min())
print("Latest:", carpark_snapshots.max())

print("\nSnapshots per Singapore calendar date:")
carpark_local = carpark_snapshots.dt.tz_convert("Asia/Singapore")

print(
    carpark_local.dt.date.value_counts().sort_index().describe()
)

# Rainfall coverage
rainfall_times = pd.to_datetime(
    rainfall_2609["timestamp"],
    errors="coerce"
)

print("\nRAINFALL COVERAGE")
print("Unique stations:", rainfall_2609["station_id"].nunique())
print("Unique timestamps:", rainfall_times.nunique())
print("Rows per station:")
print(rainfall_2609.groupby("station_id").size().describe())

print("\nObservations per station per date:")
rainfall_local = rainfall_times.dt.tz_convert("Asia/Singapore")

rainfall_daily = (
    rainfall_2609.assign(local_date=rainfall_local.dt.date)
    .groupby(["station_id", "local_date"])
    .agg(
        observation_count=("timestamp", "size"),
        unique_timestamps=("timestamp", "nunique")
    )
)

print(rainfall_daily.describe())
print("\nStation-days with fewer than 288 unique timestamps:",
      (rainfall_daily["unique_timestamps"] < 288).sum())

CARPARK SNAPSHOT COVERAGE
Unique snapshots: 720
Earliest: 2026-09-01 00:00:37+08:00
Latest: 2026-09-30 23:00:36+08:00

Snapshots per Singapore calendar date:
count       30.000000
mean     58189.733333
std         27.711486
min      58176.000000
25%      58176.000000
50%      58176.000000
75%      58183.500000
max      58272.000000
Name: count, dtype: float64

RAINFALL COVERAGE
Unique stations: 89
Unique timestamps: 8640
Rows per station:
count      89.000000
mean     8555.561798
std       519.318127
min      3810.000000
25%      8635.000000
50%      8639.000000
75%      8640.000000
max      8640.000000
dtype: float64

Observations per station per date:
       observation_count  unique_timestamps
count        2654.000000        2654.000000
mean          286.904672         286.904672
std            10.727212          10.727212
min            66.000000          66.000000
25%           288.000000         288.000000
50%           288.000000         288.000000
75%           288.000000      

## Solving the issues
- Duplicates
- Invalid records
- Unique timestamp records

### Duplicates

In [13]:
carpark_observation_key = [
    "carpark_number",
    "lot_type",
    "snapshot_time"
]

duplicate_mask = carpark2609.duplicated(
    subset=carpark_observation_key,
    keep=False
)

print("Duplicate observation rows:", duplicate_mask.sum())
print(
    "Duplicate observation groups:",
    carpark2609.duplicated(
        subset=carpark_observation_key
    ).sum()
)

Duplicate observation rows: 0
Duplicate observation groups: 0


In [14]:
# Count unique five-minute timestamps per station per Singapore date
rainfall_check = rainfall_2609.copy()

rainfall_check["timestamp_local"] = pd.to_datetime(
    rainfall_check["timestamp"],
    errors="coerce"
)

rainfall_check["local_date"] = (
    rainfall_check["timestamp_local"].dt.date
)

station_daily_coverage = (
    rainfall_check
    .groupby(["station_id", "local_date"])["timestamp_local"]
    .nunique()
    .rename("observed_slots")
    .reset_index()
)

# Build the complete station-date grid, including station-days with zero rows
all_stations = rainfall_2609["station_id"].dropna().unique()
all_dates = pd.date_range("2026-09-01", "2026-09-30").date

full_grid = pd.MultiIndex.from_product(
    [all_stations, all_dates],
    names=["station_id", "local_date"]
).to_frame(index=False)

station_daily_coverage = full_grid.merge(
    station_daily_coverage,
    on=["station_id", "local_date"],
    how="left"
)

station_daily_coverage["observed_slots"] = (
    station_daily_coverage["observed_slots"]
    .fillna(0)
    .astype(int)
)

station_daily_coverage["coverage_pct"] = (
    station_daily_coverage["observed_slots"] / 288 * 100
)

station_daily_coverage["eligible_for_ranking"] = (
    station_daily_coverage["observed_slots"] >= 260
)

print("Total expected station-days:", len(station_daily_coverage))
print(
    "Eligible station-days:",
    station_daily_coverage["eligible_for_ranking"].sum()
)
print(
    "Partial station-days below 90%:",
    (~station_daily_coverage["eligible_for_ranking"]).sum()
)

print("\nCoverage distribution:")
print(
    station_daily_coverage["observed_slots"]
    .value_counts()
    .sort_index()
)

print("\nStation-days with lowest coverage:")
display(
    station_daily_coverage
    .sort_values("observed_slots")
    .head(20)
)

Total expected station-days: 2670
Eligible station-days: 2631
Partial station-days below 90%: 39

Coverage distribution:
observed_slots
0        16
66        1
113       1
140       1
142       1
148       1
149       1
156       1
160       4
199       1
202       1
209       1
211       2
216       2
217       1
221       1
227       1
246       1
252       1
272       1
273       1
274       3
275       4
277       1
278       2
279       2
280       5
281       2
282       7
283       6
284       8
285       4
286      18
287      84
288    2483
Name: count, dtype: int64

Station-days with lowest coverage:


,station_id,local_date,observed_slots,coverage_pct,eligible_for_ranking
2651,S900,2026-09-12,0,0.000000,False
2654,S900,2026-09-15,0,0.000000,False
2655,S900,2026-09-16,0,0.000000,False
2650,S900,2026-09-11,0,0.000000,False
2649,S900,2026-09-10,0,0.000000,False
2648,S900,2026-09-09,0,0.000000,False
2653,S900,2026-09-14,0,0.000000,False
2647,S900,2026-09-08,0,0.000000,False
2645,S900,2026-09-06,0,0.000000,False
2644,S900,2026-09-05,0,0.000000,False


### standardising timestamps

In [15]:
carpark = carpark2609.copy()
rainfall = rainfall_2609.copy()
bus_od = od_bus2609.copy()
bus_stops = bus_stops2609.copy()

# Standardize column names
for df in [carpark, rainfall, bus_od, bus_stops]:
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

# Preserve original timestamp strings for audit purposes
carpark["snapshot_time_raw"] = carpark["snapshot_time"]
carpark["update_datetime_raw"] = carpark["update_datetime"]
rainfall["timestamp_raw"] = rainfall["timestamp"]

# Parse timestamps
# Offset-aware timestamps are converted to UTC for consistent comparison.
carpark["snapshot_time"] = pd.to_datetime(
    carpark["snapshot_time"],
    errors="coerce",
    utc=True
)

rainfall["timestamp"] = pd.to_datetime(
    rainfall["timestamp"],
    errors="coerce",
    utc=True
)

# update_datetime has no timezone offset.
# Parse it, but do not assign a timezone until the source convention is confirmed.
carpark["update_datetime"] = pd.to_datetime(
    carpark["update_datetime"],
    errors="coerce"
)

### Prepare rainfall daily coverage

In [16]:
# Convert rainfall to numeric
rainfall["rainfall_mm"] = pd.to_numeric(
    rainfall["rainfall_mm"],
    errors="coerce"
)

# Validate rainfall values
invalid_rainfall = (
    rainfall["rainfall_mm"].isna() |
    (rainfall["rainfall_mm"] < 0) |
    rainfall["timestamp"].isna()
)

rainfall_quarantine = rainfall[invalid_rainfall].copy()
rainfall_quarantine["quarantine_reason"] = np.select(
    [
        rainfall_quarantine["rainfall_mm"].isna(),
        rainfall_quarantine["rainfall_mm"] < 0,
        rainfall_quarantine["timestamp"].isna()
    ],
    [
        "INVALID_RAINFALL_VALUE",
        "NEGATIVE_RAINFALL",
        "INVALID_TIMESTAMP"
    ],
    default="INVALID_RAINFALL_RECORD"
)

rainfall_clean = rainfall[~invalid_rainfall].copy()

# Convert timestamps to Singapore local time for calendar-day grouping
rainfall_clean["timestamp_local"] = (
    rainfall_clean["timestamp"]
    .dt.tz_convert("Asia/Singapore")
)

rainfall_clean["local_date"] = (
    rainfall_clean["timestamp_local"].dt.date
)

# Calculate observed slots for each station-day
rainfall_daily = (
    rainfall_clean
    .groupby(["station_id", "local_date"])
    .agg(
        observed_rainfall_mm=("rainfall_mm", "sum"),
        observed_slots=("timestamp", "nunique")
    )
    .reset_index()
)

# Add missing station-days using the expected station-date grid
stations = rainfall_clean["station_id"].unique()
expected_timestamps = pd.date_range(
    "2026-09-01 00:00:00",
    "2026-09-30 23:55:00",
    freq="5min",
    tz="Asia/Singapore"
)

expected_slot_keys = pd.MultiIndex.from_product(
    [stations, expected_timestamps],
    names=["station_id", "timestamp_local"]
)
observed_slot_keys = pd.MultiIndex.from_frame(
    rainfall_clean[["station_id", "timestamp_local"]].drop_duplicates()
)
rainfall_missing_slots = expected_slot_keys.difference(
    observed_slot_keys,
    sort=False
).to_frame(index=False)

dates = pd.date_range(
    "2026-09-01",
    "2026-09-30"
).date

expected_grid = pd.MultiIndex.from_product(
    [stations, dates],
    names=["station_id", "local_date"]
).to_frame(index=False)

rainfall_daily = expected_grid.merge(
    rainfall_daily,
    on=["station_id", "local_date"],
    how="left"
)

rainfall_daily["observed_slots"] = (
    rainfall_daily["observed_slots"]
    .fillna(0)
    .astype(int)
)

rainfall_daily["coverage_pct"] = (
    rainfall_daily["observed_slots"] / 288 * 100
)

rainfall_daily["eligible_for_ranking"] = (
    rainfall_daily["observed_slots"] >= 260
)

print("Eligible station-days:",
      rainfall_daily["eligible_for_ranking"].sum())

print("Incomplete station-days:",
      (~rainfall_daily["eligible_for_ranking"]).sum())
print("Missing five-minute slots:", len(rainfall_missing_slots))

Eligible station-days: 2631
Incomplete station-days: 39
Missing five-minute slots: 7515


## BUS OD VALIDATION

In [17]:
# BUS OD VALIDATION

# Check required fields
bus_od_required = [
    "year_month",
    "day_type",
    "time_per_hour",
    "pt_type",
    "origin_pt_code",
    "destination_pt_code",
    "total_trips"
]

print("Missing required columns:",
      set(bus_od_required) - set(bus_od.columns))

# Validate categories
print("\nDay types:")
print(bus_od["day_type"].value_counts(dropna=False))

print("\nPublic transport types:")
print(bus_od["pt_type"].value_counts(dropna=False))

# Validate hour
print("\nInvalid hours:",
      (~bus_od["time_per_hour"].between(0, 23)).sum())

# Validate trips
print("Negative trips:",
      (bus_od["total_trips"] < 0).sum())

# Validate month format
valid_month = bus_od["year_month"].str.fullmatch(
    r"\d{4}-(0[1-9]|1[0-2])"
)

print("Invalid year_month values:", (~valid_month).sum())

Missing required columns: set()

Day types:
day_type
WEEKDAY             3414011
WEEKENDS/HOLIDAY    2449909
Name: count, dtype: Int64

Public transport types:
pt_type
BUS    5863920
Name: count, dtype: Int64

Invalid hours: 0
Negative trips: 0
Invalid year_month values: 0


In [18]:
# BUS STOP REFERENTIAL INTEGRITY

stop_ids = set(bus_stops["busstopcode"].dropna())

origin_exists = bus_od["origin_pt_code"].isin(stop_ids)
destination_exists = bus_od["destination_pt_code"].isin(stop_ids)

print("Origin codes not found:", (~origin_exists).sum())
print("Destination codes not found:", (~destination_exists).sum())

unmatched_origin = (
    bus_od.loc[~origin_exists, "origin_pt_code"]
    .value_counts()
)

unmatched_destination = (
    bus_od.loc[~destination_exists, "destination_pt_code"]
    .value_counts()
)

print("\nUnmatched origins:")
print(unmatched_origin.head(20))

print("\nUnmatched destinations:")
print(unmatched_destination.head(20))

Origin codes not found: 3542
Destination codes not found: 4500

Unmatched origins:
origin_pt_code
67759    3317
44711     120
70372      35
67689      31
67708      29
07522      10
Name: count, dtype: Int64

Unmatched destinations:
destination_pt_code
67759    3463
67689     844
44711     156
67708      23
07522      14
Name: count, dtype: Int64


In [19]:
bus_od = bus_od_raw.copy()

# Numeric conversion
bus_od["TIME_PER_HOUR"] = pd.to_numeric(
    bus_od["TIME_PER_HOUR"], errors="coerce"
)

bus_od["TOTAL_TRIPS"] = pd.to_numeric(
    bus_od["TOTAL_TRIPS"], errors="coerce"
)

# Validation rules
bus_od_rules = {
    "MISSING_REQUIRED_VALUE": (
        bus_od[
            [
                "YEAR_MONTH",
                "DAY_TYPE",
                "TIME_PER_HOUR",
                "PT_TYPE",
                "ORIGIN_PT_CODE",
                "DESTINATION_PT_CODE",
                "TOTAL_TRIPS"
            ]
        ].isna().any(axis=1)
    ),

    "INVALID_YEAR_MONTH": (
        ~bus_od["YEAR_MONTH"].str.fullmatch(
            r"\d{4}-(0[1-9]|1[0-2])", na=False
        )
    ),

    "INVALID_DAY_TYPE": (
        ~bus_od["DAY_TYPE"].isin(
            ["WEEKDAY", "WEEKENDS/HOLIDAY"]
        )
    ),

    "INVALID_PT_TYPE": (
        bus_od["PT_TYPE"] != "BUS"
    ),

    "INVALID_HOUR": (
        ~bus_od["TIME_PER_HOUR"].between(0, 23)
        | (bus_od["TIME_PER_HOUR"] % 1 != 0)
    ),

    "INVALID_TRIP_COUNT": (
        bus_od["TOTAL_TRIPS"].isna()
        | (bus_od["TOTAL_TRIPS"] < 0)
        | (bus_od["TOTAL_TRIPS"] % 1 != 0)
    ),

    "DUPLICATE_ANALYTICAL_KEY": bus_od.duplicated(
        subset=[
            "YEAR_MONTH", "DAY_TYPE", "TIME_PER_HOUR", "PT_TYPE",
            "ORIGIN_PT_CODE", "DESTINATION_PT_CODE"
        ],
        keep=False
    )
}

bus_od_clean, bus_od_quarantine = validate_and_quarantine(
    bus_od,
    bus_od_rules
)

bus_od_key = [
    "YEAR_MONTH",
    "DAY_TYPE",
    "TIME_PER_HOUR",
    "PT_TYPE",
    "ORIGIN_PT_CODE",
    "DESTINATION_PT_CODE"
]

bus_od_duplicates = bus_od_clean.duplicated(
    subset=bus_od_key,
    keep=False
)

print("Duplicate OD records:", bus_od_duplicates.sum())

Duplicate OD records: 0


### Clean bus stop metadata

In [20]:
bus_stops = bus_stops_raw.copy()

bus_stops["Latitude"] = pd.to_numeric(
    bus_stops["Latitude"], errors="coerce"
)

bus_stops["Longitude"] = pd.to_numeric(
    bus_stops["Longitude"], errors="coerce"
)

bus_stops_rules = {
    "MISSING_REQUIRED_VALUE": (
        bus_stops[
            ["BusStopCode", "RoadName", "Description",
             "Latitude", "Longitude"]
        ].isna().any(axis=1)
    ),

    "INVALID_COORDINATES": (
        ~bus_stops["Latitude"].between(-90, 90)
        | ~bus_stops["Longitude"].between(-180, 180)
    ),

    "DUPLICATE_BUS_STOP_CODE": bus_stops.duplicated(
        subset=["BusStopCode"],
        keep=False
    )
}

bus_stops_clean, bus_stops_quarantine = validate_and_quarantine(
    bus_stops,
    bus_stops_rules
)

# Check metadata key uniqueness.
bus_stop_duplicates = bus_stops_clean.duplicated(
    subset=["BusStopCode"],
    keep=False
)

print("Duplicate bus stop records:", bus_stop_duplicates.sum())

Duplicate bus stop records: 0


In [21]:
rainfall = rainfall_raw.copy()

rainfall["rainfall_mm"] = pd.to_numeric(
    rainfall["rainfall_mm"], errors="coerce"
)

rainfall["timestamp"] = pd.to_datetime(
    rainfall["timestamp"],
    errors="coerce",
    utc=True
)

rainfall_rules = {
    "MISSING_REQUIRED_VALUE": (
        rainfall[
            ["timestamp", "station_id", "rainfall_mm"]
        ].isna().any(axis=1)
    ),

    "INVALID_RAINFALL": (
        rainfall["rainfall_mm"].isna()
        | (rainfall["rainfall_mm"] < 0)
    ),

    "MISALIGNED_TIMESTAMP": (
        rainfall["timestamp"] != rainfall["timestamp"].dt.floor("5min")
    ),

    "DUPLICATE_STATION_TIMESTAMP": rainfall.duplicated(
        subset=["station_id", "timestamp"],
        keep=False
    )
}

rainfall_clean, rainfall_quarantine = validate_and_quarantine(
    rainfall,
    rainfall_rules
)

rainfall_key = ["station_id", "timestamp"]

rainfall_duplicates = rainfall_clean.duplicated(
    subset=rainfall_key,
    keep=False
)

print("Duplicate rainfall records:", rainfall_duplicates.sum())

Duplicate rainfall records: 0


### rainfall timestamp validation

In [22]:
# RAINFALL SLOT ALIGNMENT

rainfall_times = rainfall_clean["timestamp"].dt.tz_convert(
    "Asia/Singapore"
)

print("Minute distribution:")
print(rainfall_times.dt.minute.value_counts().sort_index())

print("\nSecond distribution:")
print(rainfall_times.dt.second.value_counts().sort_index())

# Expected five-minute minute values
valid_minutes = {0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55}

invalid_alignment = (
    ~rainfall_times.dt.minute.isin(valid_minutes) |
    (rainfall_times.dt.second != 0)
)

print("\nMisaligned observations:", invalid_alignment.sum())

Minute distribution:
timestamp
0     63479
5     63434
10    63462
15    63453
20    63462
25    63444
30    63459
35    63459
40    63459
45    63391
50    63455
55    63488
Name: count, dtype: int64

Second distribution:
timestamp
0    761445
Name: count, dtype: int64

Misaligned observations: 0


### clean carpark data

In [23]:
carpark = carpark_raw.copy()

carpark["snapshot_time"] = pd.to_datetime(
    carpark["snapshot_time"],
    errors="coerce",
    utc=True
)

carpark["update_datetime"] = pd.to_datetime(
    carpark["update_datetime"],
    errors="coerce"
)

carpark["lots_available"] = pd.to_numeric(
    carpark["lots_available"], errors="coerce"
)

carpark["total_lots"] = pd.to_numeric(
    carpark["total_lots"], errors="coerce"
)

carpark_rules = {
    "MISSING_REQUIRED_VALUE": (
        carpark[
            [
                "snapshot_time",
                "carpark_number",
                "lot_type",
                "lots_available",
                "total_lots",
                "update_datetime"
            ]
        ].isna().any(axis=1)
    ),

    "INVALID_CAPACITY": (
        carpark["total_lots"].isna()
        | (carpark["total_lots"] <= 0)
    ),

    "INVALID_AVAILABLE_LOTS": (
        carpark["lots_available"].isna()
        | (carpark["lots_available"] < 0)
    ),

    "AVAILABLE_EXCEEDS_CAPACITY": (
        carpark["lots_available"] > carpark["total_lots"]
    ),
    "DUPLICATE_OBSERVATION_KEY": carpark.duplicated(
        subset=["carpark_number", "lot_type", "snapshot_time"],
        keep=False
    )
}

carpark_clean, carpark_quarantine = validate_and_quarantine(
    carpark,
    carpark_rules
)
carpark_key = [
    "carpark_number",
    "lot_type",
    "snapshot_time"
]

print("Original records:", len(carpark))
print("Quarantined records:", len(carpark_quarantine))
print("Valid records:", len(carpark_clean))
print("Duplicate records quarantined:", carpark_quarantine["quarantine_reason"].str.contains("DUPLICATE_OBSERVATION_KEY").sum())

Original records: 1745692
Quarantined records: 3908
Valid records: 1741784
Duplicate records quarantined: 0


### Carpark source freshness

We previously found some carparks whose source timestamps did not change across all 720 snapshots. That means the API repeatedly returned a value whose underlying source update time was unchanged.

We should measure how widespread this is

In [24]:
# Convert snapshot time to Singapore local time
snapshot_sg = (
    carpark_clean["snapshot_time"]
    .dt.tz_convert("Asia/Singapore")
    .dt.tz_localize(None)
)

carpark_clean["update_datetime_age"] = (
    snapshot_sg - carpark_clean["update_datetime"]
)

print(carpark_clean["update_datetime_age"].describe())

print("\nSource updates older than:")
for hours in [1, 6, 12, 24, 72, 168]:
    count = (
        carpark_clean["update_datetime_age"]
        > pd.Timedelta(hours=hours)
    ).sum()

    print(f"{hours:>3} hours: {count:,}")

count                      1741784
mean     0 days 09:14:15.639806083
std      8 days 22:33:47.848022753
min              -1 days +16:22:12
25%                0 days 00:01:17
50%                0 days 00:02:40
75%                0 days 00:11:22
max              333 days 09:15:36
Name: update_datetime_age, dtype: object

Source updates older than:
  1 hours: 14,330
  6 hours: 6,662
 12 hours: 5,989
 24 hours: 5,551
 72 hours: 4,870
168 hours: 4,179


### type c dataset

In [25]:
# LOT TYPE C PROFILE

carpark_c = carpark_clean[
    carpark_clean["lot_type"] == "C"
].copy()

print("Lot type C records:", len(carpark_c))

print("\nUnique carparks:", carpark_c["carpark_number"].nunique())

print("\nCapacity summary:")
print(carpark_c["total_lots"].describe())

print("\nAvailability summary:")
print(carpark_c["lots_available"].describe())

carpark_c["occupancy_ratio"] = (
    carpark_c["total_lots"] - carpark_c["lots_available"]
) / carpark_c["total_lots"]

print("\nOccupancy ratio summary:")
print(carpark_c["occupancy_ratio"].describe())

print("\nOccupancy ratio outside [0, 1]:",
      (
          ~carpark_c["occupancy_ratio"].between(0, 1)
      ).sum())

Lot type C records: 1440824

Unique carparks: 2005

Capacity summary:
count    1.440824e+06
mean     3.535704e+02
std      2.764870e+02
min      1.000000e+00
25%      1.580000e+02
50%      3.100000e+02
75%      4.790000e+02
max      2.754000e+03
Name: total_lots, dtype: float64

Availability summary:
count    1.440824e+06
mean     2.108815e+02
std      2.132921e+02
min      0.000000e+00
25%      6.100000e+01
50%      1.630000e+02
75%      2.930000e+02
max      2.754000e+03
Name: lots_available, dtype: float64

Occupancy ratio summary:
count    1.440824e+06
mean     4.652829e-01
std      2.425540e-01
min      0.000000e+00
25%      2.930233e-01
50%      4.380000e-01
75%      6.106195e-01
max      1.000000e+00
Name: occupancy_ratio, dtype: float64

Occupancy ratio outside [0, 1]: 0


In [26]:
# Convert timestamps to Singapore time for export.
rainfall_clean["timestamp"] = (
    rainfall_clean["timestamp"]
    .dt.tz_convert("Asia/Singapore")
)

carpark_clean["snapshot_time"] = (
    carpark_clean["snapshot_time"]
    .dt.tz_convert("Asia/Singapore")
)

# Remove temporary analysis columns from exported OD data.
bus_od_export = bus_od_clean.drop(
    columns=["origin_matched", "destination_matched"],
    errors="ignore"
)

# Export cleaned datasets.
bus_od_export.to_csv(
    OUTPUT_DIR / "bus_od_clean.csv",
    index=False
)

rainfall_missing_slots.to_csv(
    REPORT_DIR / "rainfall_missing_slots.csv",
    index=False
)

bus_stops_clean.to_csv(
    OUTPUT_DIR / "bus_stops_clean.csv",
    index=False
)

rainfall_clean.drop(
    columns=["local_date"], errors="ignore"
).to_csv(
    OUTPUT_DIR / "rainfall_clean.csv",
    index=False
)

carpark_clean.drop(
    columns=["local_date"], errors="ignore"
).to_csv(
    OUTPUT_DIR / "carpark_clean.csv",
    index=False
)

# Export quarantined records.
bus_od_quarantine.to_csv(
    QUARANTINE_DIR / "bus_od_quarantine.csv",
    index=False
)

bus_stops_quarantine.to_csv(
    QUARANTINE_DIR / "bus_stops_quarantine.csv",
    index=False
)

rainfall_quarantine.to_csv(
    QUARANTINE_DIR / "rainfall_quarantine.csv",
    index=False
)

carpark_quarantine.to_csv(
    QUARANTINE_DIR / "carpark_quarantine.csv",
    index=False
)

print("Cleaning and export completed.")

Cleaning and export completed.


In [27]:
quality_summary = pd.DataFrame([
    {
        "dataset": "bus_od",
        "original_records": len(bus_od_raw),
        "clean_records": len(bus_od_clean),
        "quarantined_records": len(bus_od_quarantine)
    },
    {
        "dataset": "bus_stops",
        "original_records": len(bus_stops_raw),
        "clean_records": len(bus_stops_clean),
        "quarantined_records": len(bus_stops_quarantine)
    },
    {
        "dataset": "rainfall",
        "original_records": len(rainfall_raw),
        "clean_records": len(rainfall_clean),
        "quarantined_records": len(rainfall_quarantine)
    },
    {
        "dataset": "carpark",
        "original_records": len(carpark_raw),
        "clean_records": len(carpark_clean),
        "quarantined_records": len(carpark_quarantine)
    }
])

quality_summary["retention_percentage"] = (
    quality_summary["clean_records"]
    / quality_summary["original_records"]
    * 100
)

quality_summary.to_csv(
    REPORT_DIR / "data_quality_summary.csv",
    index=False
)

display(quality_summary)

,dataset,original_records,clean_records,quarantined_records,retention_percentage
0,bus_od,5863920,5863920,0,100.000000
1,bus_stops,5209,5209,0,100.000000
2,rainfall,761445,761445,0,100.000000
3,carpark,1745692,1741784,3908,99.776135


In [28]:
print("BUS OD")
print(bus_od_clean.shape)
print(bus_od_quarantine.shape)

print("\nBUS STOPS")
print(bus_stops_clean.shape)
print(bus_stops_quarantine.shape)

print("\nRAINFALL")
print(rainfall_clean.shape)
print(rainfall_quarantine.shape)

print("\nCARPARK")
print(carpark_clean.shape)
print(carpark_quarantine.shape)

print("\nDUPLICATE KEYS")
print("Bus OD:", bus_od_clean.duplicated(subset=bus_od_key).sum())
print("Bus stops:", bus_stops_clean.duplicated(subset=["BusStopCode"]).sum())
print("Rainfall:", rainfall_clean.duplicated(subset=rainfall_key).sum())
print("Carpark:", carpark_clean.duplicated(subset=carpark_key).sum())


BUS OD
(5863920, 7)
(0, 8)

BUS STOPS
(5209, 5)
(0, 6)

RAINFALL
(761445, 6)
(0, 7)

CARPARK
(1741784, 7)
(3908, 7)

DUPLICATE KEYS
Bus OD: 0
Bus stops: 0
Rainfall: 0
Carpark: 0


### calculating sha256

In [31]:
import hashlib
from pathlib import Path

def calculate_sha256(filepath):
    sha256 = hashlib.sha256()

    with open(filepath, "rb") as file:
        for chunk in iter(lambda: file.read(4096), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

In [32]:
DATA_DIR = Path("CA1_Datasets")

for filepath in DATA_DIR.glob("*"):
    if filepath.is_file():
        print(filepath.name)
        print(calculate_sha256(filepath))
        print()

bus_od_202609.zip
447be2570d7fdc3b2530911278f70f3d32ddff88ec971421b1fe716199d138dc

bus_stops.csv
711889094d5b7ca4588d23643beae1e2afe9233bf56c85a33761e2c97ee13baf

carpark_20260901_20260930.csv
9b4af13d990b9c702f241a8d4d4cdd3154a6ac66a9258bf0ef084600a27f0664

carpark_20260901_20260930.progress
be2a021bf9f459d4ab8311b98a6c5a74761e3776b620142fe6eb978b596e1711

carpark_20260901_20260930_pre_resume_backup.csv
4248ba2faa3f293e841de31aa9bf66948b84e69ac18d488cb301323bd0c2dc39

origin_destination_bus_202609.csv
1b45454a55e04051aa7ad1f678b209c0b4ca96e3f9ce9e38ea96223a5d7c6030

rainfall_20260901_20260930.csv
e61a74d98bac6b4b1ede00547e7338511538378c163caedc68e5b0ea26eea04d

